# Week 61, Day 4: ExitStack

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w061d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from contextlib import ExitStack

log = []
with ExitStack() as stack:
    for name in ["a", "b", "c"]:
        log.append(f"open {name}")
        stack.callback(log.append, f"close {name}")
    log.append("work")
print(log)

In [ ]:
from contextlib import ExitStack
import heapq

for name, lines in [("a.txt", "1\n4\n9\n"), ("b.txt", "2\n3\n10\n"), ("c.txt", "5\n")]:
    with open(name, "w") as f:
        f.write(lines)

def merged_numbers(paths):
    with ExitStack() as stack:
        files = [stack.enter_context(open(p)) for p in paths]
        streams = [(int(line) for line in f) for f in files]
        return list(heapq.merge(*streams))

print(merged_numbers(["a.txt", "b.txt", "c.txt"]))

In [ ]:
from contextlib import ExitStack
import os

with open("scratch.tmp", "w") as f:
    f.write("temporary data")

settings = {"verbose": False}
with ExitStack() as stack:
    stack.callback(os.remove, "scratch.tmp")                # delete the file at the end
    old = settings["verbose"]
    settings["verbose"] = True
    stack.callback(settings.__setitem__, "verbose", old)   # restore the setting
    print("inside:", settings["verbose"], os.path.exists("scratch.tmp"))
print("after:", settings["verbose"], os.path.exists("scratch.tmp"))

In [ ]:
from contextlib import ExitStack

def run_steps(steps, log):
    with ExitStack() as stack:
        for name, do, undo in steps:
            do()
            log.append(f"did {name}")
            stack.callback(log.append, f"undid {name}")
            stack.callback(undo)
        stack.pop_all()               # all succeeded: keep everything, cancel the undos

def fail():
    raise RuntimeError("disk full")

log = []
nothing = lambda: None
try:
    run_steps([("folder", nothing, nothing), ("config", nothing, nothing), ("index", fail, nothing)], log)
except RuntimeError:
    pass
print(log)

In [ ]:
from contextlib import ExitStack

class Workspace:
    def __init__(self):
        self._stack = ExitStack()
        self.files = {}

    def open(self, path, mode="r"):
        f = self._stack.enter_context(open(path, mode))
        self.files[path] = f
        return f

    def close(self):
        self._stack.close()

    def __enter__(self):
        return self

    def __exit__(self, *exc):
        self.close()
        return False

with Workspace() as ws:
    ws.open("one.txt", "w").write("1")
    ws.open("two.txt", "w").write("2")
print([f.closed for f in ws.files.values()])

> **Key idea.** ExitStack holds any number of context managers and cleanup functions and releases them in reverse order when it closes, passing exceptions along as nested with statements would. enter_context enters a manager and returns its value; callback registers a plain function call; push registers an __exit__-style function. Register each completed step's undo as a callback, so a later failure undoes them all, newest first. pop_all moves everything to a new stack, cancelling the cleanup here: use it to keep resources (or completed work) only when every step succeeded, and hand the returned stack to whoever now owns the cleanup. Classes can keep an ExitStack attribute and close it in close() or __exit__.

### Exercise: Opening a list of files

Write `read_all(paths)`: a dict from each path to the file's text, opening all the files with **one** `ExitStack` (all open at once). Then `total_size(paths)`: the total number of characters, the same way. If any file is missing, `FileNotFoundError` must propagate and every file already opened must be closed.

In [ ]:
from contextlib import ExitStack

def read_all(paths):
    ___

def total_size(paths):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_many():\n    \"enter_context\"\n    for name, text in [(\"x.txt\", \"ab\"), (\"y.txt\", \"cde\")]:\n        with open(name, \"w\") as f:\n            f.write(text)\n    assert read_all([\"x.txt\", \"y.txt\"]) == {\"x.txt\": \"ab\", \"y.txt\": \"cde\"} and total_size([\"x.txt\", \"y.txt\"]) == 5\n    assert read_all([]) == {}\n    try:\n        read_all([\"x.txt\", \"missing.txt\"])\n        assert False\n    except FileNotFoundError:\n        pass\n    assert \"enter_context(\" in source")

### Exercise: Last in, first out

Write `cleanup_order(names)`: register a callback for each name, in order, that appends `"clean " + name` to a log, run a body that appends `"work"`, and return the log. Then `with_failure(names, fail_at)`: register callbacks for the names one at a time but raise `RuntimeError` just **before** registering the name at position `fail_at`; catch the error outside the stack and return the log.

In [ ]:
from contextlib import ExitStack

def cleanup_order(names):
    ___

def with_failure(names, fail_at):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_order():\n    \"callbacks\"\n    assert cleanup_order([\"a\", \"b\", \"c\"]) == [\"work\", \"clean c\", \"clean b\", \"clean a\"]\n    assert with_failure([\"a\", \"b\", \"c\"], 2) == [\"clean b\", \"clean a\"] and with_failure([\"a\"], 0) == []\n    assert cleanup_order([]) == [\"work\"]")

### Exercise: Steps that undo themselves

Write `apply_changes(data, changes)`: `data` is a dict, `changes` a list of `(key, value)` pairs; set each one in turn, registering on an `ExitStack` an undo that restores the old value (or removes the key if it was new). A value of `None` is invalid: raise `ValueError(f"bad value for {key}")` when you reach it, which must undo every change already made. If every change succeeds, keep them (use `pop_all`). Return the number of changes applied.

In [ ]:
from contextlib import ExitStack

_MISSING = object()

def apply_changes(data, changes):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_undo():\n    \"rollback with callbacks\"\n    d = {\"a\": 1, \"b\": 2}\n    assert apply_changes(d, [(\"a\", 10), (\"c\", 3)]) == 2 and d == {\"a\": 10, \"b\": 2, \"c\": 3}\n    try:\n        apply_changes(d, [(\"a\", 99), (\"new\", 5), (\"b\", None), (\"z\", 1)])\n        assert False\n    except ValueError as e:\n        assert str(e) == \"bad value for b\"\n    assert d == {\"a\": 10, \"b\": 2, \"c\": 3}\n    assert apply_changes(d, []) == 0\n    assert \"pop_all(\" in source and \"callback(\" in source")

### Exercise: Handing files to the caller

Write `open_for_writing(paths)`: open every path for writing; if they all open, return `(files, stack)`, where `stack` is an `ExitStack` (from `pop_all`) that closes them all; if any open fails, close the ones already opened and let the error propagate. Then `write_report(paths, lines)`: use it to write every line to every file, closing them all at the end with `with stack:`.

In [ ]:
from contextlib import ExitStack

def open_for_writing(paths):
    ___

def write_report(paths, lines):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import os\ndef test_handoff():\n    \"pop_all\"\n    files, stack = open_for_writing([\"r1.txt\", \"r2.txt\"])\n    assert all(not f.closed for f in files)\n    with stack:\n        files[0].write(\"x\")\n    assert all(f.closed for f in files)\n    os.makedirs(\"locked_dir\", exist_ok=True)\n    try:\n        open_for_writing([\"ok.txt\", \"locked_dir\"])\n        assert False\n    except (IsADirectoryError, PermissionError, OSError):\n        pass\n    write_report([\"p.txt\", \"q.txt\"], [\"one\", \"two\"])\n    with open(\"q.txt\") as f:\n        assert f.read() == \"one\\ntwo\\n\"")

### Exercise: A class that owns its files

Write `Logbook()`: a context manager class with its own `ExitStack`. `add(name)` opens `name + ".log"` for appending (keeping it open, managed by the stack) and returns nothing; `write(name, text)` writes a line to that log; `close()` closes every log; `open_logs()` is the sorted list of names whose files are still open. Using it in `with` closes everything at the end.

In [ ]:
from contextlib import ExitStack

class Logbook:
    def __init__(self):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_owner():\n    \"a stack attribute\"\n    with Logbook() as book:\n        book.add(\"errors\")\n        book.add(\"access\")\n        book.write(\"errors\", \"disk full\")\n        book.write(\"access\", \"ada logged in\")\n        assert book.open_logs() == [\"access\", \"errors\"]\n    assert book.open_logs() == []\n    with open(\"errors.log\") as f:\n        assert f.read() == \"disk full\\n\"\n    b2 = Logbook()\n    b2.add(\"x\")\n    b2.close()\n    assert b2.open_logs() == []")

### Bug hunt: A stack that's never closed

The `ExitStack` is created but not used in a `with` (and never closed), so its files stay open and this prints `[False, False]`. Use it in a `with` statement so it prints `[True, True]`.

In [ ]:
from contextlib import ExitStack

for name in ("m.txt", "n.txt"):
    with open(name, "w") as f:
        f.write(name)

stack = ExitStack()
files = [stack.enter_context(open(p)) for p in ("m.txt", "n.txt")]
texts = [f.read() for f in files]
print([f.closed for f in files])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"with the stack\"\n    expect_output(\"[True, True]\")")

### Bug hunt: A callback called too early

`callback` takes the **function** and its arguments separately. Writing `stack.callback(log.append("closed"))` calls `append` immediately (logging too early) and registers its result, `None`, which fails when the stack closes with `TypeError`. Pass the function and its argument separately so this prints `['work', 'closed']`.

In [ ]:
from contextlib import ExitStack

log = []
try:
    with ExitStack() as stack:
        stack.callback(log.append("closed"))
        log.append("work")
except TypeError:
    log.append("TypeError")
print(log)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"function and arguments\"\n    expect_output(\"['work', 'closed']\")")

### Bug hunt: Closed before the caller gets them

Returning from inside the `with ExitStack()` closes everything on the stack, so the caller receives closed files and writing fails with `ValueError`. Return `stack.pop_all()` with the files, so this prints `written`.

In [ ]:
from contextlib import ExitStack

def open_two():
    with ExitStack() as stack:
        a = stack.enter_context(open("a1.txt", "w"))
        b = stack.enter_context(open("b1.txt", "w"))
        return a, b

a, b = open_two()
try:
    a.write("x")
    print("written")
except ValueError:
    print("closed already")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"pop_all\"\n    expect_output(\"written\")")

### Puzzle: Opening a list of files

Arrange the lines that read every file in `paths` with all of them open at once. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the order callbacks run in: "first in, first out" or "last in, first out"
q = ___   # what enter_context returns: "the manager" or "what __enter__ returns"
r = ___   # True or False: pop_all leaves the original stack empty
s = ___   # True or False: if the third of five opens fails, the first two files stay open
t = ___   # the ExitStack method that runs everything now, without a with statement, as a string

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"last in, first out\" and q == \"what __enter__ returns\" and r is True, \"p, q and r\"\n    assert s is False and t == \"close\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a deployment with rollbacks

A deployment is a list of steps. Each step is a dict with a `"name"`, a `"do"` function, an optional `"undo"` function, and an optional `"uses"`: a list of context managers the step needs while it runs (opened before `do`, closed straight after). Write `deploy(steps)`:

- Run the steps in order. Each step's managers are entered (with their own `ExitStack`) just for that step.
- After each successful step, register its undo (if any) on the **main** stack.
- If a step fails, undo every completed step, newest first, and re-raise the error.
- If every step succeeds, cancel the undos.
- Return a report: a list of strings: `"ran NAME"` for each completed step, `"failed NAME: ERROR_TYPE"` for the failing one, and `"undid NAME"` for each undo that ran, in the order things happened. The report must be available even when the deployment fails, so `deploy` takes the `report` list as an argument, fills it in as it goes, and returns it on success.

In [ ]:
from contextlib import ExitStack

def deploy(steps, report):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from contextlib import contextmanager\ndef test_deploy():\n    \"ExitStack in two layers\"\n    events = []\n    @contextmanager\n    def connection(name):\n        events.append(f\"connect {name}\")\n        try:\n            yield name\n        finally:\n            events.append(f\"disconnect {name}\")\n    def step(name, fail=False, undo=True, uses=()):\n        def do():\n            events.append(f\"do {name}\")\n            if fail:\n                raise RuntimeError(name)\n        s = {\"name\": name, \"do\": do, \"uses\": [connection(u) for u in uses]}\n        if undo:\n            s[\"undo\"] = lambda: events.append(f\"undo {name}\")\n        return s\n    report = []\n    assert deploy([step(\"db\", uses=[\"server\"]), step(\"files\", undo=False), step(\"cache\")], report) == [\"ran db\", \"ran files\", \"ran cache\"]\n    assert events == [\"connect server\", \"do db\", \"disconnect server\", \"do files\", \"do cache\"]\n    events.clear()\n    report = []\n    try:\n        deploy([step(\"db\"), step(\"files\", undo=False), step(\"cache\", fail=True, uses=[\"redis\"]), step(\"never\")], report)\n        assert False\n    except RuntimeError:\n        pass\n    assert report == [\"ran db\", \"ran files\", \"failed cache: RuntimeError\", \"undid db\"]\n    assert events == [\"do db\", \"do files\", \"connect redis\", \"do cache\", \"disconnect redis\", \"undo db\"]")

## Warm-up

Cleanups registered on the stack run when the `with` ends, **last in, first out**: like nested `with` statements, whatever was set up last is released first.

## Any number of managers

`stack.enter_context(manager)` enters a context manager (calling its `__enter__`, and returning what it returns, like `as`) and remembers to exit it when the stack closes. So opening a list of files is a loop:

```python
with ExitStack() as stack:
    files = [stack.enter_context(open(p)) for p in paths]
    ...                          # every file is open here
                                 # every file is closed here, even after an error
```

If opening the third file fails, the first two are still closed properly, because they're already on the stack. The stack passes exceptions to each manager's `__exit__` exactly as nested `with` statements would (Monday's challenge), including suppression.

## Cleanup functions

Not everything that needs undoing is a context manager. `stack.callback(func, *args, **kwargs)` registers a plain function call to run at the end: deleting a temporary file, restoring a setting, printing a summary. Callbacks and entered managers share one stack, so they all unwind together in reverse order.

`stack.push(exit_function)` registers a function with `__exit__`'s signature, `(exc_type, exc, tb)`, that sees the exception and may suppress it. It's rarely needed; `callback` covers most uses.

## Undoing what went before

A multi-step job (set up a user: create a folder, write a config file, add them to a list) should leave nothing half-done if a later step fails. With a stack: after each step succeeds, register its **undo** as a callback. If a later step raises, leaving the `with` runs the undos for every completed step, newest first. If everything succeeds, the stack would undo them too, unless you **cancel** the callbacks first: that's what `pop_all` is for.

## All or nothing with pop_all

`stack.pop_all()` moves everything on the stack to a **new** `ExitStack` and returns it, leaving the original empty, so the original's `with` ends without cleaning anything up. That's how a function can set up several resources, keep them all if every setup succeeded, and hand them to its caller, while still closing the ones it managed to open if any setup failed:

```python
def open_all(paths):
    with ExitStack() as stack:
        files = [stack.enter_context(open(p)) for p in paths]
        return files, stack.pop_all()      # the caller now owns the cleanup
```

If the third `open` fails, the `with` closes the first two. If all succeed, `pop_all` transfers them, and the caller later uses `with returned_stack:` or calls `returned_stack.close()`.

## A stack in a class

An object that opens things over its lifetime (a workspace with several files, a test fixture, a client with connections) can keep its own `ExitStack` as an attribute: `self._stack.enter_context(...)` whenever it opens something, and a `close()` method that calls `self._stack.close()`. Make the class a context manager too (`__enter__` returns self, `__exit__` calls `close()`), and users get `with Workspace() as ws:` with everything cleaned up at the end.

## Practice

## Quiz

## Daily challenge

## Before you go

Think of a job with several steps (installing something, moving into a new flat, setting up a group trip) and list each step's undo. Which order would they be undone in? Tomorrow: managing resources safely, with locks, temporary files and timeouts.